# 1. Despligue de un cluster de Vault en EKS

## 1.1. Conexión con clúster de EKS

## AWS KMS auto-unseal con IRSA

Creamos una clave KMS dedicada y un rol IRSA limitado al service account `vault:vault`. Vault usará las credenciales temporales proporcionadas por IRSA; no se almacenan access keys en Kubernetes.

In [ ]:
%%bash
set -euo pipefail

doormat login -f
eval "$(doormat aws -a aws_jose.merchan_test export)"
mkdir -p /tmp/vault
WORKDIR=/tmp/vault

AWS_REGION=eu-central-1
EKS_CLUSTER_NAME=eks-infra-dev
VAULT_NAMESPACE=vault
VAULT_SERVICE_ACCOUNT=vault
VAULT_KMS_ALIAS=alias/vault-auto-unseal
VAULT_KMS_POLICY_NAME=vault-kms-auto-unseal
VAULT_IRSA_ROLE_NAME=vault-kms-auto-unseal
AWS_ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)

# IRSA requires an IAM OIDC provider for the EKS cluster.
OIDC_ISSUER=$(aws eks describe-cluster --name ${EKS_CLUSTER_NAME} --region ${AWS_REGION} --query 'cluster.identity.oidc.issuer' --output text)
OIDC_PROVIDER=${OIDC_ISSUER#https://}
OIDC_PROVIDER_ARN=arn:aws:iam::${AWS_ACCOUNT_ID}:oidc-provider/${OIDC_PROVIDER}
if ! aws iam get-open-id-connect-provider --open-id-connect-provider-arn ${OIDC_PROVIDER_ARN} >/dev/null 2>&1; then
  aws iam create-open-id-connect-provider \
    --url ${OIDC_ISSUER} \
    --client-id-list sts.amazonaws.com \
    --tags Key=Name,Value=${EKS_CLUSTER_NAME}-irsa >/dev/null
fi

# Create a dedicated symmetric KMS key once and address it through a stable alias.
if ! aws kms describe-key --key-id ${VAULT_KMS_ALIAS} --region ${AWS_REGION} >/dev/null 2>&1; then
  VAULT_KMS_KEY_ID=$(aws kms create-key \
    --region ${AWS_REGION} \
    --description "Vault auto-unseal" \
    --tags TagKey=Name,TagValue=vault-auto-unseal \
    --query KeyMetadata.KeyId \
    --output text)
  aws kms create-alias \
    --region ${AWS_REGION} \
    --alias-name ${VAULT_KMS_ALIAS} \
    --target-key-id ${VAULT_KMS_KEY_ID}
fi
VAULT_KMS_KEY_ARN=$(aws kms describe-key --key-id ${VAULT_KMS_ALIAS} --region ${AWS_REGION} --query KeyMetadata.Arn --output text)

cat > ${WORKDIR}/vault-kms-policy.json <<EOF
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Action": ["kms:Encrypt", "kms:Decrypt", "kms:DescribeKey"],
    "Resource": "${VAULT_KMS_KEY_ARN}"
  }]
}
EOF

VAULT_KMS_POLICY_ARN=arn:aws:iam::${AWS_ACCOUNT_ID}:policy/${VAULT_KMS_POLICY_NAME}
if ! aws iam get-policy --policy-arn ${VAULT_KMS_POLICY_ARN} >/dev/null 2>&1; then
  aws iam create-policy \
    --policy-name ${VAULT_KMS_POLICY_NAME} \
    --policy-document file://${WORKDIR}/vault-kms-policy.json >/dev/null
else
  for VERSION_ID in $(aws iam list-policy-versions --policy-arn ${VAULT_KMS_POLICY_ARN} --query 'Versions[?IsDefaultVersion==`false`].VersionId' --output text); do
    aws iam delete-policy-version --policy-arn ${VAULT_KMS_POLICY_ARN} --version-id ${VERSION_ID}
  done
  aws iam create-policy-version \
    --policy-arn ${VAULT_KMS_POLICY_ARN} \
    --policy-document file://${WORKDIR}/vault-kms-policy.json \
    --set-as-default >/dev/null
fi

cat > ${WORKDIR}/vault-irsa-trust-policy.json <<EOF
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Principal": {"Federated": "${OIDC_PROVIDER_ARN}"},
    "Action": "sts:AssumeRoleWithWebIdentity",
    "Condition": {
      "StringEquals": {
        "${OIDC_PROVIDER}:aud": "sts.amazonaws.com",
        "${OIDC_PROVIDER}:sub": "system:serviceaccount:${VAULT_NAMESPACE}:${VAULT_SERVICE_ACCOUNT}"
      }
    }
  }]
}
EOF

if ! aws iam get-role --role-name ${VAULT_IRSA_ROLE_NAME} >/dev/null 2>&1; then
  aws iam create-role \
    --role-name ${VAULT_IRSA_ROLE_NAME} \
    --description "IRSA role for Vault AWS KMS auto-unseal" \
    --assume-role-policy-document file://${WORKDIR}/vault-irsa-trust-policy.json >/dev/null
else
  aws iam update-assume-role-policy \
    --role-name ${VAULT_IRSA_ROLE_NAME} \
    --policy-document file://${WORKDIR}/vault-irsa-trust-policy.json
fi
aws iam attach-role-policy \
  --role-name ${VAULT_IRSA_ROLE_NAME} \
  --policy-arn ${VAULT_KMS_POLICY_ARN}

echo "KMS key: ${VAULT_KMS_KEY_ARN}"
echo "IRSA role: arn:aws:iam::${AWS_ACCOUNT_ID}:role/${VAULT_IRSA_ROLE_NAME}"

KMS key: arn:aws:kms:eu-central-1:492487827579:key/3e67d9b0-40c0-4d3f-a7c8-1d97e56e9250
IRSA role: arn:aws:iam::492487827579:role/vault-kms-auto-unseal


In [2]:
# Hashi only
!doormat login -f

import os
import subprocess

# Import the credentials produced by doormat into the notebook kernel.
result = subprocess.run(
    ["bash", "-lc", 'eval "$(doormat aws -a aws_jose.merchan_test export)" && env -0'],
    check=True,
    capture_output=True,
)
for entry in result.stdout.split(b"\0"):
    if entry.startswith(b"AWS_") and b"=" in entry:
        key, value = entry.split(b"=", 1)
        os.environ[key.decode()] = value.decode()

os.environ["AWS_REGION"] = "eu-central-1"

!aws eks update-kubeconfig --region eu-central-1 --name eks-infra-dev
!aws eks create-access-entry --region eu-central-1 --cluster-name eks-infra-dev --principal-arn arn:aws:iam::492487827579:role/aws_jose.merchan_test-developer
!aws eks associate-access-policy --region eu-central-1 --cluster-name eks-infra-dev --principal-arn arn:aws:iam::492487827579:role/aws_jose.merchan_test-developer --policy-arn arn:aws:eks::aws:cluster-access-policy/AmazonEKSClusterAdminPolicy --access-scope type=cluster


INFO[0001] logging into doormat...                      
INFO[0002] successfully logged into doormat!            
Updated context arn:aws:eks:eu-central-1:492487827579:cluster/eks-infra-dev in /Users/jose/.kube/config

aws: [ERROR]: An error occurred (ResourceInUseException) when calling the CreateAccessEntry operation: The specified access entry resource is already in use on this cluster.
{
    "clusterName": "eks-infra-dev",
    "principalArn": "arn:aws:iam::492487827579:role/aws_jose.merchan_test-developer",
    "associatedAccessPolicy": {
        "policyArn": "arn:aws:eks::aws:cluster-access-policy/AmazonEKSClusterAdminPolicy",
        "accessScope": {
            "type": "cluster",
            "namespaces": []
        },
        "associatedAt": "2026-07-20T15:17:02.491000+02:00",
        "modifiedAt": "2026-07-23T11:00:43.849000+02:00"
    }
}


In [44]:
!kubectl get nodes

NAME                                           STATUS   ROLES    AGE     VERSION
ip-10-1-4-91.eu-central-1.compute.internal     Ready    <none>   3d23h   v1.33.13-eks-8f14419
ip-10-1-42-43.eu-central-1.compute.internal    Ready    <none>   3d23h   v1.33.13-eks-8f14419
ip-10-1-89-121.eu-central-1.compute.internal   Ready    <none>   3d23h   v1.33.13-eks-8f14419


Añadimos el repo de HashiCorp para a continuación poder instalar Vault vía helm

In [45]:
%%bash
helm repo add hashicorp https://helm.releases.hashicorp.com
helm repo update

"hashicorp" already exists with the same configuration, skipping
Hang tight while we grab the latest from your chart repositories...
...Successfully got an update from the "secrets-store-csi-driver" chart repository
...Successfully got an update from the "hashicorp" chart repository
...Successfully got an update from the "kspm-helm-charts" chart repository
...Successfully got an update from the "bitnami" chart repository
Update Complete. ⎈Happy Helming!⎈


Definimos una serie de variable de entorno con las que trabajaremos a continuación

In [46]:
%env WORKDIR=/tmp/vault
%env VAULT_K8S_NAMESPACE=vault
%env VAULT_HELM_RELEASE_NAME=vault
%env VAULT_SERVICE_NAME=vault-internal 
%env K8S_CLUSTER_NAME=cluster.local 

env: WORKDIR=/tmp/vault
env: VAULT_K8S_NAMESPACE=vault
env: VAULT_HELM_RELEASE_NAME=vault
env: VAULT_SERVICE_NAME=vault-internal
env: K8S_CLUSTER_NAME=cluster.local


Creamos un directorio temporal donde se almacenaran los certificados, claves de Shamir y root token de Vault

In [47]:
%%bash
rm -rf /tmp/vault
mkdir /tmp/vault

Vamos a generar certificados para Vault usando Let's Encrypt. Para tal fin usaremos certbot para obtener y validar el certificado haciendo uso de la zona disponible en la cuenta de AWS

In [48]:
! kubectl create namespace $VAULT_K8S_NAMESPACE

namespace/vault created


In [49]:
%%bash
set -euo pipefail

export AWS_REGION="${AWS_REGION:-eu-central-1}"
export AWS_DEFAULT_REGION="${AWS_REGION}"
ROUTE53_ZONE_ID="Z001600238RLTE1X7639D"
ZONE_NAME=$(aws route53 get-hosted-zone \
  --id "${ROUTE53_ZONE_ID}" \
  --query 'HostedZone.Name' \
  --output text | sed 's/\.$//')
VAULT_FQDN="vault.${ZONE_NAME}"
LETSENCRYPT_DIR="${WORKDIR}/letsencrypt"
CERTBOT_ACCOUNT_ARGS=(--register-unsafely-without-email)
if [[ -n "${LETSENCRYPT_EMAIL:-}" ]]; then
  CERTBOT_ACCOUNT_ARGS=(--email "${LETSENCRYPT_EMAIL}")
fi

mkdir -p "${LETSENCRYPT_DIR}"
cat > "${WORKDIR}/vault-fqdn.env" <<EOF
ROUTE53_ZONE_ID=${ROUTE53_ZONE_ID}
VAULT_FQDN=${VAULT_FQDN}
EOF

# DNS-01 proves domain ownership through Route 53; Vault does not need to be reachable yet.
docker run --rm \
  -e AWS_ACCESS_KEY_ID \
  -e AWS_SECRET_ACCESS_KEY \
  -e AWS_SESSION_TOKEN \
  -e AWS_REGION \
  -e AWS_DEFAULT_REGION \
  -v "${LETSENCRYPT_DIR}:/etc/letsencrypt" \
  certbot/dns-route53:latest certonly \
  --dns-route53 \
  --non-interactive \
  --agree-tos \
  "${CERTBOT_ACCOUNT_ARGS[@]}" \
  --keep-until-expiring \
  -d "${VAULT_FQDN}"

CERTIFICATE_DIR="${LETSENCRYPT_DIR}/live/${VAULT_FQDN}"
test -s "${CERTIFICATE_DIR}/fullchain.pem"
test -s "${CERTIFICATE_DIR}/privkey.pem"
test -s "${CERTIFICATE_DIR}/chain.pem"
openssl x509 -in "${CERTIFICATE_DIR}/fullchain.pem" -noout -subject -issuer -dates
SYSTEM_ROOTS=$(mktemp)
trap 'rm -f "${SYSTEM_ROOTS}"' EXIT
security find-certificate -a -p /System/Library/Keychains/SystemRootCertificates.keychain > "${SYSTEM_ROOTS}"
openssl verify \
  -CAfile "${SYSTEM_ROOTS}" \
  -untrusted "${CERTIFICATE_DIR}/chain.pem" \
  "${CERTIFICATE_DIR}/cert.pem"

kubectl create secret generic vault-ha-tls \
  -n "${VAULT_K8S_NAMESPACE}" \
  --from-file=vault.key="${CERTIFICATE_DIR}/privkey.pem" \
  --from-file=vault.crt="${CERTIFICATE_DIR}/fullchain.pem" \
  --from-file=vault.ca="${CERTIFICATE_DIR}/chain.pem" \
  --dry-run=client -o yaml | kubectl apply -f -

Saving debug log to /var/log/letsencrypt/letsencrypt.log


Account registered.
Requesting a certificate for vault.jose-merchan.sbx.hashidemos.io

Successfully received certificate.
Certificate is saved at: /etc/letsencrypt/live/vault.jose-merchan.sbx.hashidemos.io/fullchain.pem
Key is saved at:         /etc/letsencrypt/live/vault.jose-merchan.sbx.hashidemos.io/privkey.pem
This certificate expires on 2026-10-22.
These files will be updated when the certificate renews.
NEXT STEPS:
- The certificate will need to be renewed before it expires. Certbot can automatically renew the certificate in the background, but you may need to take steps to enable that functionality. See https://certbot.org/renewal-setup for instructions.

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
If you like Certbot, please consider supporting our work by:
 * Donating to ISRG / Let's Encrypt:   https://letsencrypt.org/donate
 * Donating to EFF:                    https://eff.org/donate-le
- - - - - - - - - - - - - - - - - - - - - - - - - - -

Aplicaremos una licencia [Enterprise](https://developer.hashicorp.com/vault/docs/platform/k8s/helm/enterprise) al cluster de Vault

In [50]:
%%bash
secret=$(cat vault.hclic)
kubectl create secret generic vault-ent-license --from-literal="license=${secret}" -n $VAULT_K8S_NAMESPACE

secret/vault-ent-license created


Creamos de forma idempotente el `imagePullSecret` para descargar la imagen privada desde Docker Hub. La credencial se obtiene del almacén seguro de Docker Desktop y no se guarda en el notebook.

In [51]:
%%bash
set -euo pipefail
DOCKERHUB_CREDENTIAL=$(printf '%s' 'https://index.docker.io/v1/' | docker-credential-desktop get)
DOCKERHUB_USERNAME=$(jq -r '.Username' <<<"${DOCKERHUB_CREDENTIAL}")
DOCKERHUB_TOKEN=$(jq -r '.Secret' <<<"${DOCKERHUB_CREDENTIAL}")
test -n "${DOCKERHUB_USERNAME}" && test -n "${DOCKERHUB_TOKEN}"
kubectl create secret docker-registry dockerhub-josemerchan \
  -n "${VAULT_K8S_NAMESPACE}" \
  --docker-server=https://index.docker.io/v1/ \
  --docker-username="${DOCKERHUB_USERNAME}" \
  --docker-password="${DOCKERHUB_TOKEN}" \
  --dry-run=client -o yaml | kubectl apply -f -
unset DOCKERHUB_CREDENTIAL DOCKERHUB_TOKEN

secret/dockerhub-josemerchan created


Creamos el fichero de configuración con el que instalar Vault. Instalaremos lo siguiente:
* Un clúster HA compuesto por 6 nodos con auto_join.
* Cluster con TLS usando certificados firmados por la CA de Kubernetes.
* Tanto los certificados como las licencias se montaran en un volumen
* Distribuiremos los 6 servidores uniformemente entre 3 zonas de disponibilidad (2 por zona) y habilitaremos Vault Enterprise Autopilot Redundancy Zones. Requiere Vault Helm 0.34.0+, Kubernetes 1.35+ con PodTopologyLabelsAdmission y nodos etiquetados con `topology.kubernetes.io/zone`.
* Instalaremos Vault agent injector
* Montaremos el plugin Enterprise de Oracle `0.14.1+ent` y Oracle Instant Client `19.26` en todos los pods de Vault.

In [52]:
%%bash
set -euo pipefail
source "${WORKDIR}/vault-fqdn.env"
VAULT_IRSA_ROLE_ARN=arn:aws:iam::$(aws sts get-caller-identity --query Account --output text):role/vault-kms-auto-unseal
cat > ${WORKDIR}/overrides.yaml <<EOF
global:
   enabled: true
   tlsDisable: false
   imagePullSecrets:
      - name: dockerhub-josemerchan
   # openshift: true

csi:
   enabled: true
   image:
      repository: "docker.io/hashicorp/vault-csi-provider"

injector:
   enabled: true
   image:
      repository: docker.io/hashicorp/vault-k8s
   agentImage:
      repository: docker.io/hashicorp/vault

# Supported log levels include: trace, debug, info, warn, error
logLevel: "trace" # Set to trace for initial troubleshooting, info for normal operation

server:
   serviceAccount:
      create: true
      name: vault
      annotations:
         eks.amazonaws.com/role-arn: ${VAULT_IRSA_ROLE_ARN}
         eks.amazonaws.com/sts-regional-endpoints: "true"
   service:
      port: 443
      targetPort: 8200
      type: "LoadBalancer"
      annotations:
         service.beta.kubernetes.io/aws-load-balancer-scheme: internet-facing
   image:
      repository: docker.io/josemerchan/vault-oracle-init
      tag: vault2.0.3-ent-oracle0.14.1-ic19.26-ic23.26.3@sha256:b8d4367fe99ab97cb9d0437a32ba2558d750303057c27d78da3be233e7c27f10
   enterpriseLicense:
      secretName: vault-ent-license
   extraEnvironmentVars:
      VAULT_CACERT: /vault/userconfig/vault-ha-tls/vault.ca
      VAULT_TLSCERT: /vault/userconfig/vault-ha-tls/vault.crt
      VAULT_TLSKEY: /vault/userconfig/vault-ha-tls/vault.key
      LD_LIBRARY_PATH: /opt/oracle/instantclient_23_26
      ORACLE_HOME: /opt/oracle/instantclient_23_26
   volumes:
      - name: userconfig-vault-ha-tls
        secret:
         defaultMode: 420
         secretName: vault-ha-tls
   volumeMounts:
      - mountPath: /vault/userconfig/vault-ha-tls
        name: userconfig-vault-ha-tls
        readOnly: true
   standalone:
      enabled: false
   affinity: ""
   # Spread six Vault servers evenly: two per availability zone.
   topologySpreadConstraints:
      - maxSkew: 1
        minDomains: 3
        topologyKey: topology.kubernetes.io/zone
        whenUnsatisfiable: DoNotSchedule
        labelSelector:
           matchLabels:
              app.kubernetes.io/name: vault
              app.kubernetes.io/instance: vault
              component: server
   ha:
      enabled: true
      replicas: 6
      raft:
         enabled: true
         setNodeId: true
         redundancyZones:
            enabled: true
         config: |
            ui = true
            api_addr = "https://${VAULT_FQDN}"
            plugin_directory = "/vault/plugins"
            seal "awskms" {
               region     = "eu-central-1"
               kms_key_id = "alias/vault-auto-unseal"
            }

            listener "tcp" {
               tls_disable = 0 
               address = "[::]:8200"
               cluster_address = "[::]:8201"
               tls_cert_file = "/vault/userconfig/vault-ha-tls/vault.crt"
               tls_key_file  = "/vault/userconfig/vault-ha-tls/vault.key"
               tls_client_ca_file = "/vault/userconfig/vault-ha-tls/vault.ca"
            }

            storage "raft" {
               path = "/vault/data"
               autopilot_redundancy_zone = "VAULT_REDUNDANCY_ZONE"
            
               retry_join {
                  auto_join             = "provider=k8s namespace=vault label_selector=\"component=server,app.kubernetes.io/name=vault\""
                  auto_join_scheme      = "https"
                  leader_ca_cert_file   = "/vault/userconfig/vault-ha-tls/vault.ca"
                  leader_tls_servername = "${VAULT_FQDN}"
               }
            
            }
            telemetry {
               disable_hostname = true
               prometheus_retention_time = "12h"
            }
            disable_mlock = true
            service_registration "kubernetes" {}

# Vault UI exposed through an internet-facing AWS load balancer.
ui:
   enabled: true
   serviceType: "LoadBalancer"
   annotations:
      service.beta.kubernetes.io/aws-load-balancer-scheme: internet-facing
   serviceNodePort: null
   externalPort: 443
   targetPort: 8200

   
EOF


Finalmente usando el values.yaml instalamos Vault

In [53]:
%%bash
set -euo pipefail

helm upgrade --install \
  -n "$VAULT_K8S_NAMESPACE" \
  "$VAULT_HELM_RELEASE_NAME" \
  hashicorp/vault \
  --version 0.34.0 \
  -f "${WORKDIR}/overrides.yaml" \
  --server-side=false \
  --wait \
  --timeout 10m

Release "vault" does not exist. Installing it now.
NAME: vault
LAST DEPLOYED: Fri Jul 24 13:02:49 2026
NAMESPACE: vault
STATUS: deployed
REVISION: 1
DESCRIPTION: Install complete
NOTES:
Thank you for installing HashiCorp Vault!

Now that you have deployed Vault, you should look over the docs on using
Vault with Kubernetes available here:

https://developer.hashicorp.com/vault/docs


Your release is named vault. To learn more about the release, try:

  $ helm status vault
  $ helm get manifest vault


## Asociate the vault-active service as CNAME to an static A record (vault.jose-merchan.sbx.hashidemos.io)

In [54]:
%%bash
set -euo pipefail
source "${WORKDIR}/vault-fqdn.env"
export AWS_REGION="${AWS_REGION:-eu-central-1}"

# The chart publishes the hostname before the ELB API necessarily exposes its alias zone.
VAULT_ACTIVE_LB_HOST=""
for attempt in $(seq 1 40); do
  VAULT_ACTIVE_LB_HOST=$(kubectl get service vault-active \
    -n "${VAULT_K8S_NAMESPACE}" \
    -o jsonpath='{.status.loadBalancer.ingress[0].hostname}')
  if [ -n "${VAULT_ACTIVE_LB_HOST}" ]; then
    break
  fi
  sleep 15
done
if [ -z "${VAULT_ACTIVE_LB_HOST}" ]; then
  echo "vault-active LoadBalancer hostname is not ready" >&2
  exit 1
fi

LB_ZONE_ID=""
for attempt in $(seq 1 40); do
  # Kubernetes commonly provisions an NLB/ALB, which uses the ELBv2 API.
  LB_ZONE_ID=$(aws elbv2 describe-load-balancers \
    --region "${AWS_REGION}" \
    --query "LoadBalancers[?DNSName=='${VAULT_ACTIVE_LB_HOST}'].CanonicalHostedZoneId | [0]" \
    --output text)
  if [ -n "${LB_ZONE_ID}" ] && [ "${LB_ZONE_ID}" != "None" ]; then
    break
  fi

  # Retain compatibility with a Classic Load Balancer service implementation.
  LB_ZONE_ID=$(aws elb describe-load-balancers \
    --region "${AWS_REGION}" \
    --query "LoadBalancerDescriptions[?DNSName=='${VAULT_ACTIVE_LB_HOST}'].CanonicalHostedZoneNameID | [0]" \
    --output text)
  if [ -n "${LB_ZONE_ID}" ] && [ "${LB_ZONE_ID}" != "None" ]; then
    break
  fi
  sleep 15
done
if [ -z "${LB_ZONE_ID}" ] || [ "${LB_ZONE_ID}" = "None" ]; then
  echo "Load balancer ${VAULT_ACTIVE_LB_HOST} is not yet available through the AWS ELB APIs" >&2
  exit 1
fi

cat > "${WORKDIR}/vault-active-route53.json" <<EOF
{
  "Comment": "Route Vault active FQDN to its Kubernetes load balancer",
  "Changes": [{
    "Action": "UPSERT",
    "ResourceRecordSet": {
      "Name": "${VAULT_FQDN}",
      "Type": "A",
      "AliasTarget": {
        "HostedZoneId": "${LB_ZONE_ID}",
        "DNSName": "${VAULT_ACTIVE_LB_HOST}",
        "EvaluateTargetHealth": false
      }
    }
  }]
}
EOF

CHANGE_ID=$(aws route53 change-resource-record-sets \
  --hosted-zone-id "${ROUTE53_ZONE_ID}" \
  --change-batch "file://${WORKDIR}/vault-active-route53.json" \
  --query 'ChangeInfo.Id' \
  --output text)
aws route53 wait resource-record-sets-changed --id "${CHANGE_ID}"
echo "${VAULT_FQDN} now aliases ${VAULT_ACTIVE_LB_HOST}"

vault.jose-merchan.sbx.hashidemos.io now aliases aa880be785acc4ec0a1a02d7e7edde74-1160947374.eu-central-1.elb.amazonaws.com


Verifiquemos la instalación

In [55]:
! kubectl get events -n vault

LAST SEEN   TYPE      REASON                   OBJECT                                       MESSAGE
108s        Normal    WaitForFirstConsumer     persistentvolumeclaim/data-vault-0           waiting for first consumer to be created before binding
108s        Normal    Provisioning             persistentvolumeclaim/data-vault-0           External provisioner is provisioning volume for claim "vault/data-vault-0"
108s        Normal    ExternalProvisioning     persistentvolumeclaim/data-vault-0           Waiting for a volume to be created either by the external provisioner 'ebs.csi.aws.com' or manually by the system administrator. If volume creation is delayed, please verify that the provisioner is running and correctly registered.
106s        Normal    ProvisioningSucceeded    persistentvolumeclaim/data-vault-0           Successfully provisioned volume pvc-db5a1b01-8fb2-4260-be86-1547c9ed0439
108s        Normal    WaitForFirstConsumer     persistentvolumeclaim/data-vault-1           wait

In [56]:
! kubectl -n $VAULT_K8S_NAMESPACE get pods #--watch

NAME                                    READY   STATUS    RESTARTS   AGE
vault-0                                 0/1     Running   0          112s
vault-1                                 0/1     Running   0          112s
vault-2                                 0/1     Running   0          112s
vault-3                                 0/1     Running   0          112s
vault-4                                 0/1     Running   0          112s
vault-5                                 0/1     Running   0          112s
vault-agent-injector-65fcfc7599-6m9jq   1/1     Running   0          112s
vault-csi-provider-45b8p                2/2     Running   0          112s
vault-csi-provider-kpvjs                2/2     Running   0          112s
vault-csi-provider-q4wzl                2/2     Running   0          112s


El chart crea también los servicios `vault`, `vault-active` y `vault-standby`. Al definir `server.service.type: LoadBalancer`, Helm expone esos servicios externamente y `vault-active` queda gestionado dentro del release.

In [57]:
%%bash
kubectl get service -n ${VAULT_K8S_NAMESPACE} vault vault-active vault-standby vault-ui

NAME            TYPE           CLUSTER-IP       EXTERNAL-IP                                                                  PORT(S)                        AGE
vault           LoadBalancer   172.20.1.146     a07f6df1b94a74f0ebb681e942e12f17-112340075.eu-central-1.elb.amazonaws.com    443:31571/TCP,8201:31711/TCP   117s
vault-active    LoadBalancer   172.20.135.23    aa880be785acc4ec0a1a02d7e7edde74-1160947374.eu-central-1.elb.amazonaws.com   443:31736/TCP,8201:32630/TCP   117s
vault-standby   LoadBalancer   172.20.102.67    a275e08001b694388af2115dace53666-601283673.eu-central-1.elb.amazonaws.com    443:30554/TCP,8201:30144/TCP   117s
vault-ui        LoadBalancer   172.20.145.221   a6c93bae233d34e4cb47ee23a8f83f79-2061485969.eu-central-1.elb.amazonaws.com   443:30545/TCP                  117s


Una vez instalado Vault procedemos a su inicialización con AWS KMS auto-unseal. Vault generará una **recovery key** en lugar de claves de unseal. En producción deben usarse varias recovery shares con un threshold adecuado.

In [60]:
%%bash
set -e
umask 077
kubectl exec -n "$VAULT_K8S_NAMESPACE" vault-0 -- \
  env VAULT_SKIP_VERIFY=true vault operator init \
    -recovery-shares=1 \
    -recovery-threshold=1 \
    -format=json > "${WORKDIR}/cluster-keys.json"
chmod 600 "${WORKDIR}/cluster-keys.json"

Registramos el plugin Enterprise de Oracle y habilitamos el database secrets engine. La operación es idempotente.

In [61]:
%%bash
set -euo pipefail
ROOT_TOKEN=$(jq -r '.root_token' ${WORKDIR}/cluster-keys.json)
vault_exec=(kubectl exec -n "$VAULT_K8S_NAMESPACE" vault-0 -- env VAULT_SKIP_VERIFY=true VAULT_TOKEN="$ROOT_TOKEN" vault)
if ! "${vault_exec[@]}" token lookup >/dev/null 2>&1; then
    echo "ERROR: el root token local no es válido para este clúster; se cancela el registro." >&2
    exit 1
fi
"${vault_exec[@]}" plugin register \
    -version=v0.14.1+ent \
    database vault-plugin-database-oracle
if ! "${vault_exec[@]}" secrets list -format=json | jq -e 'has("database/")' >/dev/null; then
    "${vault_exec[@]}" secrets enable database
fi
"${vault_exec[@]}" plugin list -detailed database | grep -E 'NAME|vault-plugin-database-oracle'

Success! Registered plugin: vault-plugin-database-oracle
Success! Enabled the database secrets engine at: database/
vault-plugin-database-oracle         database    v0.14.1+ent             false        n/a


Guardamos la recovery key en un gestor seguro. Los pods se desellarán automáticamente mediante KMS e IRSA.

In [62]:
%%bash
jq -r ".recovery_keys_b64[]" ${WORKDIR}/cluster-keys.json

xppeDYw64cPhGRrfof0ZwVb1QrQS06a0ZRoso+NP5nU=


Nodo 0

In [64]:
!kubectl exec -n $VAULT_K8S_NAMESPACE vault-0 -- vault status --tls-skip-verify

Key                      Value
---                      -----
Seal Type                awskms
Recovery Seal Type       shamir
Initialized              true
Sealed                   false
Total Recovery Shares    1
Threshold                1
Version                  2.0.3+ent
Build Date               2026-06-16T21:32:56Z
Storage Type             raft
Cluster Name             vault-cluster-2b342670
Cluster ID               73d734d5-fbbb-559d-8efa-4ecfcc664b3f
Removed From Cluster     false
HA Enabled               true
HA Cluster               https://vault-0.vault-internal:8201
HA Mode                  active
Active Since             2026-07-24T11:09:33.231042016Z
Raft Committed Index     243
Raft Applied Index       243
Last WAL                 90


Nodo 1

In [65]:
!kubectl exec -n $VAULT_K8S_NAMESPACE vault-1 -- vault status --tls-skip-verify

Key                                    Value
---                                    -----
Seal Type                              awskms
Recovery Seal Type                     shamir
Initialized                            true
Sealed                                 false
Total Recovery Shares                  1
Threshold                              1
Version                                2.0.3+ent
Build Date                             2026-06-16T21:32:56Z
Storage Type                           raft
Cluster Name                           vault-cluster-2b342670
Cluster ID                             73d734d5-fbbb-559d-8efa-4ecfcc664b3f
Removed From Cluster                   false
HA Enabled                             true
HA Cluster                             https://vault-0.vault-internal:8201
HA Mode                                standby
Active Node Address                    https://10.1.76.24:8200
Performance Standby Node               true
Performance Standby Last Remote WAL   

Nodo 2

In [66]:
!kubectl exec -n $VAULT_K8S_NAMESPACE vault-2 -- vault status --tls-skip-verify

Key                                    Value
---                                    -----
Seal Type                              awskms
Recovery Seal Type                     shamir
Initialized                            true
Sealed                                 false
Total Recovery Shares                  1
Threshold                              1
Version                                2.0.3+ent
Build Date                             2026-06-16T21:32:56Z
Storage Type                           raft
Cluster Name                           vault-cluster-2b342670
Cluster ID                             73d734d5-fbbb-559d-8efa-4ecfcc664b3f
Removed From Cluster                   false
HA Enabled                             true
HA Cluster                             https://vault-0.vault-internal:8201
HA Mode                                standby
Active Node Address                    https://10.1.76.24:8200
Performance Standby Node               true
Performance Standby Last Remote WAL   

In [67]:
!kubectl exec -n $VAULT_K8S_NAMESPACE vault-3 -- vault status --tls-skip-verify

Key                                    Value
---                                    -----
Seal Type                              awskms
Recovery Seal Type                     shamir
Initialized                            true
Sealed                                 false
Total Recovery Shares                  1
Threshold                              1
Version                                2.0.3+ent
Build Date                             2026-06-16T21:32:56Z
Storage Type                           raft
Cluster Name                           vault-cluster-2b342670
Cluster ID                             73d734d5-fbbb-559d-8efa-4ecfcc664b3f
Removed From Cluster                   false
HA Enabled                             true
HA Cluster                             https://vault-0.vault-internal:8201
HA Mode                                standby
Active Node Address                    https://10.1.76.24:8200
Performance Standby Node               true
Performance Standby Last Remote WAL   

In [68]:
!kubectl exec -n $VAULT_K8S_NAMESPACE vault-4 -- vault status --tls-skip-verify

Key                                    Value
---                                    -----
Seal Type                              awskms
Recovery Seal Type                     shamir
Initialized                            true
Sealed                                 false
Total Recovery Shares                  1
Threshold                              1
Version                                2.0.3+ent
Build Date                             2026-06-16T21:32:56Z
Storage Type                           raft
Cluster Name                           vault-cluster-2b342670
Cluster ID                             73d734d5-fbbb-559d-8efa-4ecfcc664b3f
Removed From Cluster                   false
HA Enabled                             true
HA Cluster                             https://vault-0.vault-internal:8201
HA Mode                                standby
Active Node Address                    https://10.1.76.24:8200
Performance Standby Node               true
Performance Standby Last Remote WAL   

In [69]:
!kubectl exec -n $VAULT_K8S_NAMESPACE vault-5 -- vault status --tls-skip-verify

Key                                    Value
---                                    -----
Seal Type                              awskms
Recovery Seal Type                     shamir
Initialized                            true
Sealed                                 false
Total Recovery Shares                  1
Threshold                              1
Version                                2.0.3+ent
Build Date                             2026-06-16T21:32:56Z
Storage Type                           raft
Cluster Name                           vault-cluster-2b342670
Cluster ID                             73d734d5-fbbb-559d-8efa-4ecfcc664b3f
Removed From Cluster                   false
HA Enabled                             true
HA Cluster                             https://vault-0.vault-internal:8201
HA Mode                                standby
Active Node Address                    https://10.1.76.24:8200
Performance Standby Node               true
Performance Standby Last Remote WAL   

Por último recopilamos los endpoints externos de Vault para poder operar contra el servicio `vault-active` y conservar también la URL pública de la UI.

In [70]:
%%bash
set -euo pipefail
source "${WORKDIR}/vault-fqdn.env"
REPO_ROOT="${REPO_ROOT:-$(pwd)}"
ENV_FILE="${REPO_ROOT}/.env"
PERSISTENT_CERT_DIR="${REPO_ROOT}/.vault/letsencrypt/${VAULT_FQDN}"

VAULT_ACTIVE_LB_HOST=$(kubectl get service vault-active \
   -n ${VAULT_K8S_NAMESPACE} \
   -o jsonpath='{.status.loadBalancer.ingress[0].hostname}')
if [ -z "${VAULT_ACTIVE_LB_HOST}" ]; then
   echo "The vault-active LoadBalancer hostname is not ready yet" >&2
   exit 1
fi

VAULT_UI_LB_HOST=$(kubectl get service vault-ui \
   -n ${VAULT_K8S_NAMESPACE} \
   -o jsonpath='{.status.loadBalancer.ingress[0].hostname}')
if [ -z "${VAULT_UI_LB_HOST}" ]; then
   echo "The vault-ui LoadBalancer hostname is not ready yet" >&2
   exit 1
fi

mkdir -p "${PERSISTENT_CERT_DIR}"
cp "${WORKDIR}/letsencrypt/live/${VAULT_FQDN}/chain.pem" "${PERSISTENT_CERT_DIR}/chain.pem"
chmod 600 "${PERSISTENT_CERT_DIR}/chain.pem"

umask 077
cat > "${ENV_FILE}" <<EOF
AWS_REGION=${AWS_REGION:-eu-central-1}
VAULT_ADDR=https://${VAULT_FQDN}
VAULT_UI_ADDR=https://${VAULT_UI_LB_HOST}
VAULT_TLS_SERVER_NAME=${VAULT_FQDN}
VAULT_TOKEN=$(cat $WORKDIR/cluster-keys.json | jq -r ".root_token")
VAULT_CACERT=${PERSISTENT_CERT_DIR}/chain.pem
EOF
echo "Wrote persistent Vault configuration to ${ENV_FILE}"

Wrote persistent Vault configuration to /Users/jose/Library/CloudStorage/GoogleDrive-jose.maria.merchan@gmail.com/My Drive/Demo/Mapfre_PoC_Vault/.env


In [71]:
! helm list -n vault

NAME 	NAMESPACE	REVISION	UPDATED                              	STATUS  	CHART       	APP VERSION
vault	vault    	1       	2026-07-24 13:02:49.995022 +0200 CEST	deployed	vault-0.34.0	2.0.3      


## Verifica la disponsición de los PODs

In [72]:
%%bash
kubectl get pods -n vault \
  -l app.kubernetes.io/name=vault,component=server \
  -L topology.kubernetes.io/zone

NAME      READY   STATUS    RESTARTS   AGE     ZONE
vault-0   1/1     Running   0          8m10s   eu-central-1c
vault-1   1/1     Running   0          8m10s   eu-central-1a
vault-2   1/1     Running   0          8m10s   eu-central-1b
vault-3   1/1     Running   0          8m10s   eu-central-1a
vault-4   1/1     Running   0          8m10s   eu-central-1b
vault-5   1/1     Running   0          8m10s   eu-central-1c


In [73]:
%%bash
for pod in vault-{0..5}; do
  echo -n "$pod: "
  kubectl exec -n vault "$pod" -- printenv VAULT_REDUNDANCY_ZONE
done

vault-0: eu-central-1c
vault-1: eu-central-1a
vault-2: eu-central-1b
vault-3: eu-central-1a
vault-4: eu-central-1b
vault-5: eu-central-1c


In [74]:
import os
from pathlib import Path

env_file = Path.cwd() / ".env"
if not env_file.is_file():
    raise FileNotFoundError(f"Vault environment file not found: {env_file}")

for line in env_file.read_text().splitlines():
    line = line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    key, value = line.split("=", 1)
    os.environ[key.strip()] = value.strip().strip("\"'")

In [75]:
! vault operator raft autopilot state

Healthy:                         true
Failure Tolerance:               1
Leader:                          vault-0
Voters:
   vault-0
   vault-1
   vault-2
Optimistic Failure Tolerance:    4
Servers:
   vault-0
      Name:              vault-0
      Address:           vault-0.vault-internal:8201
      Status:            leader
      Node Status:       alive
      Healthy:           true
      Last Contact:      0s
      Last Term:         3
      Last Index:        349
      Version:           2.0.3
      Upgrade Version:   2.0.3
      Redundancy Zone:   eu-central-1c
      Node Type:         zone-voter
   vault-1
      Name:              vault-1
      Address:           vault-1.vault-internal:8201
      Status:            voter
      Node Status:       alive
      Healthy:           true
      Last Contact:      3.494803406s
      Last Term:         3
      Last Index:        339
      Version:           2.0.3
      Upgrade Version:   2.0.3
      Redundancy Zone:   eu-central-1a
      

## Remove node and verify voters stays the same

In [76]:
%%bash
kubectl delete pod -n vault vault-3

sleep 20

pod "vault-3" deleted from vault namespace


In [77]:
! vault operator raft autopilot state

Healthy:                         true
Failure Tolerance:               1
Leader:                          vault-0
Voters:
   vault-0
   vault-1
   vault-2
Optimistic Failure Tolerance:    4
Servers:
   vault-0
      Name:              vault-0
      Address:           vault-0.vault-internal:8201
      Status:            leader
      Node Status:       alive
      Healthy:           true
      Last Contact:      0s
      Last Term:         3
      Last Index:        519
      Version:           2.0.3
      Upgrade Version:   2.0.3
      Redundancy Zone:   eu-central-1c
      Node Type:         zone-voter
   vault-1
      Name:              vault-1
      Address:           vault-1.vault-internal:8201
      Status:            voter
      Node Status:       alive
      Healthy:           true
      Last Contact:      495.542433ms
      Last Term:         3
      Last Index:        517
      Version:           2.0.3
      Upgrade Version:   2.0.3
      Redundancy Zone:   eu-central-1a
      

## Eliminar el entorno

Esta operación elimina el release de Helm, el namespace y sus volúmenes, el rol y la política IRSA, y programa la eliminación de la clave KMS tras 7 días. El proveedor OIDC no se elimina por defecto porque puede ser compartido por otros workloads del clúster. Cambia `CONFIRM_DESTROY` antes de ejecutar.

In [40]:
%%bash
set -euo pipefail

# Desbloqueo explícito para evitar una ejecución accidental.
CONFIRM_DESTROY="DESTROY_VAULT_DEMO" # Cambiar a: DESTROY_VAULT_DEMO
DELETE_OIDC_PROVIDER=false # true solo si este clúster no tiene ningún otro consumidor de IRSA

if [ "${CONFIRM_DESTROY}" != "DESTROY_VAULT_DEMO" ]; then
  echo "Cancelado: establece CONFIRM_DESTROY=DESTROY_VAULT_DEMO en esta celda" >&2
  exit 1
fi

doormat login -f
eval "$(doormat aws -a aws_jose.merchan_test export)"

AWS_REGION=eu-central-1
EKS_CLUSTER_NAME=eks-infra-dev
VAULT_NAMESPACE=vault
VAULT_HELM_RELEASE_NAME=vault
VAULT_KMS_ALIAS=alias/vault-auto-unseal
VAULT_KMS_POLICY_NAME=vault-kms-auto-unseal
VAULT_IRSA_ROLE_NAME=vault-kms-auto-unseal
AWS_ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)
VAULT_KMS_POLICY_ARN=arn:aws:iam::${AWS_ACCOUNT_ID}:policy/${VAULT_KMS_POLICY_NAME}

# Resolve la clave antes de eliminar el alias.
VAULT_KMS_KEY_ID=$(aws kms describe-key \
  --region ${AWS_REGION} \
  --key-id ${VAULT_KMS_ALIAS} \
  --query KeyMetadata.KeyId \
  --output text 2>/dev/null || true)

# El Service LoadBalancer se elimina con el release y AWS retirará el NLB de forma asíncrona.
helm uninstall ${VAULT_HELM_RELEASE_NAME} -n ${VAULT_NAMESPACE} --ignore-not-found
kubectl delete namespace ${VAULT_NAMESPACE} --ignore-not-found --wait=true
kubectl delete certificatesigningrequest vault.svc --ignore-not-found

# Elimina primero la asociación IAM y después sus recursos.
aws iam detach-role-policy \
  --role-name ${VAULT_IRSA_ROLE_NAME} \
  --policy-arn ${VAULT_KMS_POLICY_ARN} 2>/dev/null || true
aws iam delete-role --role-name ${VAULT_IRSA_ROLE_NAME} 2>/dev/null || true
if aws iam get-policy --policy-arn ${VAULT_KMS_POLICY_ARN} >/dev/null 2>&1; then
  for VERSION_ID in $(aws iam list-policy-versions --policy-arn ${VAULT_KMS_POLICY_ARN} --query 'Versions[?IsDefaultVersion==`false`].VersionId' --output text); do
    aws iam delete-policy-version --policy-arn ${VAULT_KMS_POLICY_ARN} --version-id ${VERSION_ID}
  done
  aws iam delete-policy --policy-arn ${VAULT_KMS_POLICY_ARN}
fi

# KMS no permite eliminación inmediata: se elimina el alias y se programa la clave a 7 días.
aws kms delete-alias --region ${AWS_REGION} --alias-name ${VAULT_KMS_ALIAS} 2>/dev/null || true
if [ -n "${VAULT_KMS_KEY_ID}" ] && [ "${VAULT_KMS_KEY_ID}" != "None" ]; then
  KEY_STATE=$(aws kms describe-key --region ${AWS_REGION} --key-id ${VAULT_KMS_KEY_ID} --query KeyMetadata.KeyState --output text)
  if [ "${KEY_STATE}" != "PendingDeletion" ]; then
    aws kms schedule-key-deletion \
      --region ${AWS_REGION} \
      --key-id ${VAULT_KMS_KEY_ID} \
      --pending-window-in-days 7
  fi
fi

# El proveedor OIDC pertenece al clúster y puede estar compartido por otros roles IRSA.
if [ "${DELETE_OIDC_PROVIDER}" = "true" ]; then
  OIDC_ISSUER=$(aws eks describe-cluster --name ${EKS_CLUSTER_NAME} --region ${AWS_REGION} --query 'cluster.identity.oidc.issuer' --output text)
  OIDC_PROVIDER=${OIDC_ISSUER#https://}
  OIDC_PROVIDER_ARN=arn:aws:iam::${AWS_ACCOUNT_ID}:oidc-provider/${OIDC_PROVIDER}
  aws iam delete-open-id-connect-provider --open-id-connect-provider-arn ${OIDC_PROVIDER_ARN}
fi

WORKDIR=/tmp/vault
rm -rf "${WORKDIR:?}"
echo "Entorno Vault eliminado. La clave KMS queda programada para eliminación en 7 días."

time="2026-07-24T12:55:30+02:00" level=info msg="logging into doormat..."
time="2026-07-24T12:55:34+02:00" level=info msg="successfully logged into doormat!"


release "vault" uninstalled
namespace "vault" deleted
{
    "KeyId": "arn:aws:kms:eu-central-1:492487827579:key/60d29b5e-a923-4bfd-85b2-8f094ca33e98",
    "DeletionDate": "2026-07-31T12:57:29.820000+02:00",
    "KeyState": "PendingDeletion",
    "PendingWindowInDays": 7
}
Entorno Vault eliminado. La clave KMS queda programada para eliminación en 7 días.
